In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
x = os.path.join(path, 'Q1_data.csv')
xx = pd.read_csv(x)


In [ ]:
print(f"Shape: {xx.shape}")
xx.head()

In [ ]:
xx.info()

In [ ]:
xx.describe()

In [ ]:
xx.drop(columns=['Order_ID'])

In [ ]:
missing_percentage = (xx.isnull().sum() / len(xx)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
def check_duplicates(xx):
  duplicates = xx.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    xx.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(xx)

In [ ]:
cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
df_clean = xx[cols].copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Courier_Experience_yrs', 'Time_of_Day','Traffic_Level','Weather'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
print("Separating target variable and features...")
X = xx.drop('Delivery_Time', axis=1)
y = xx['Delivery_Time']

print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Applying One-Hot Encoding to feature DataFrame 'X'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))

print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")
print(f"Shape of y_encoded: {y_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])


In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch

# load data
data = load_breast_cancer()
X = data.data
y = data.target

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_encoded

In [ ]:
X = X_encoded.drop("Courier_Experience_yrs_nan", axis=1).astype(float)
y = X_encoded['Courier_Experience_yrs_nan'].astype(float)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as sklearn_mse, mean_squared_error, r2_score

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

In [ ]:
x

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(xx)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

In [ ]:
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [ ]:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

In [ ]:
coeffs = {}

coeffs['Lasso'] = ['LASSO Regression'].coef_
coeffs['Ridge'] = ['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = xx.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: